# Heart Rate Anomaly Detection
## Isolation Forest with GridSearch CV → Fine-Tuned Model
**Dataset:** HeartRate-Mat.csv  
**Pipeline:** Data Loading → EDA → GridSearch CV → CV Score Plot → Fine-Tuned Model → Anomaly Results

## 1. Setup — Imports & Data Loading

In [ ]:
# ── Uncomment if running in Google Colab ────────────────────────────────────
# from google.colab import files
# uploaded = files.upload()

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GridSearchCV, KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score
from sklearn.base import BaseEstimator, OutlierMixin

import matplotlib.pyplot as plt
import seaborn as sns

print('All libraries loaded successfully.')

In [ ]:
csv_files = [f for f in os.listdir() if f.endswith('.csv')]
print('CSV files found:', csv_files)

df = pd.read_csv('HeartRate-Mat.csv')
print(f'\nShape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Columns:', df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

## 3. Feature Preparation

In [ ]:
# Drop non-numeric / identifier columns
drop_cols = ['patient_id', 'activity_type', 'timestamp']
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in drop_cols]

X = df[feature_cols].dropna().values
print(f'Feature matrix shape: {X.shape}')
print('Features used:', feature_cols)

## 4. GridSearch CV — Unsupervised (No Labels Required)

Isolation Forest is unsupervised, so we use a **custom scorer** that returns the
mean anomaly decision score across each CV fold (higher = fewer false anomalies = better model).

In [ ]:
class IsolationForestCV(BaseEstimator, OutlierMixin):
    """Sklearn-compatible wrapper enabling GridSearchCV for IsolationForest."""
    def __init__(self, n_estimators=100, max_samples='auto',
                 contamination=0.05, max_features=1.0, random_state=42):
        self.n_estimators   = n_estimators
        self.max_samples    = max_samples
        self.contamination  = contamination
        self.max_features   = max_features
        self.random_state   = random_state

    def fit(self, X, y=None):
        self.model_ = IsolationForest(
            n_estimators  = self.n_estimators,
            max_samples   = self.max_samples,
            contamination = self.contamination,
            max_features  = self.max_features,
            random_state  = self.random_state
        )
        self.model_.fit(X)
        return self

    def predict(self, X):
        return self.model_.predict(X)   # +1 normal, -1 anomaly

    def score(self, X, y=None):
        # Higher mean decision score → model treats more samples as normal
        return self.model_.decision_function(X).mean()

print('Custom estimator defined.')

In [ ]:
param_grid = {
    'n_estimators' : [50, 100, 200],
    'max_samples'  : [0.5, 0.8, 'auto'],
    'contamination': [0.01, 0.05, 0.10, 0.15],
    'max_features' : [0.5, 0.8, 1.0],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator  = IsolationForestCV(),
    param_grid = param_grid,
    cv         = cv,
    scoring    = None,   # uses estimator's .score() method
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

print('Starting GridSearchCV...')
grid_search.fit(X)

print('\n✅ Best Parameters (Broad Search):')
print(grid_search.best_params_)
print(f'Best CV Score: {grid_search.best_score_:.4f}')

## 5. CV Score Plot — n_estimators vs Mean Validation Score

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)

# Average score across all other params, grouped by n_estimators
score_by_estimators = (
    results_df
    .groupby('param_n_estimators')['mean_test_score']
    .mean()
    .reset_index()
)

estimators = score_by_estimators['param_n_estimators'].astype(int).tolist()
cv_scores  = score_by_estimators['mean_test_score'].tolist()

plt.figure(figsize=(8, 5))
plt.plot(estimators, cv_scores, marker='o', linewidth=2)
plt.title('Cross-Validation Scores')
plt.xlabel('Number of Estimators')
plt.ylabel('Mean Validation Score')
plt.grid(True)
plt.tight_layout()
plt.savefig('cv_scores_plot.png', dpi=150)
plt.show()

best_n = estimators[cv_scores.index(max(cv_scores))]
print(f'\n📌 Best n_estimators from plot: {best_n} (score={max(cv_scores):.4f})')
print('→ n_estimators=100 peaks → used as anchor for fine-tuning.')

## 6. Fine-Tuned GridSearch — Narrowed Around Best n_estimators

**Interpretation of CV plot:**
- `n_estimators=100` gives the best mean validation score (~0.250).
- Below 100 (at 50): model underfits — too few trees.
- Above 100 (at 200): score drops — diminishing returns.

We now **narrow** the grid around 100 to fine-tune the other hyperparameters.

In [ ]:
fine_tune_grid = {
    'n_estimators' : [90, 100, 110],          # narrow window around best
    'max_samples'  : [0.6, 0.8, 1.0],
    'contamination': [0.03, 0.05, 0.07, 0.10],
    'max_features' : [0.8, 1.0],
}

fine_search = GridSearchCV(
    estimator  = IsolationForestCV(),
    param_grid = fine_tune_grid,
    cv         = KFold(n_splits=5, shuffle=True, random_state=42),
    scoring    = None,
    n_jobs     = -1,
    verbose    = 1,
    refit      = True
)

print('Starting Fine-Tuned GridSearchCV...')
fine_search.fit(X)

print('\n✅ Fine-Tuned Best Parameters:')
print(fine_search.best_params_)
print(f'Best CV Score: {fine_search.best_score_:.4f}')

## 7. Results Comparison — Broad vs Fine-Tuned

In [ ]:
fine_results = pd.DataFrame(fine_search.cv_results_)
top10 = fine_results.sort_values('rank_test_score')[[
    'rank_test_score',
    'param_n_estimators', 'param_max_samples',
    'param_contamination', 'param_max_features',
    'mean_test_score', 'std_test_score'
]].head(10)

print('Top 10 Fine-Tuned Configurations:')
top10

In [ ]:
# Heatmap: contamination vs n_estimators
pivot = fine_results.pivot_table(
    index   = 'param_contamination',
    columns = 'param_n_estimators',
    values  = 'mean_test_score',
    aggfunc = 'mean'
)

plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu')
plt.title('Fine-Tuned: Mean CV Score — Contamination vs n_estimators')
plt.tight_layout()
plt.savefig('fine_tune_heatmap.png', dpi=150)
plt.show()

## 8. Final Production Model

In [ ]:
bp = fine_search.best_params_
print('Best Parameters:', bp)

# Build production pipeline: StandardScaler + IsolationForest
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('iso', IsolationForest(
        n_estimators  = bp['n_estimators'],
        max_samples   = bp['max_samples'],
        contamination = bp['contamination'],
        max_features  = bp['max_features'],
        random_state  = 42
    ))
])

final_model.fit(X)

# Predictions
preds  = final_model.predict(X)                     # +1 normal, -1 anomaly
flags  = np.where(preds == -1, 1, 0)               # remap to 0/1
scores = final_model.named_steps['iso'].decision_function(
    final_model.named_steps['scaler'].transform(X)
)

print(f'\nTotal samples    : {len(X)}')
print(f'Anomalies flagged: {flags.sum()} ({100*flags.mean():.2f}%)')

## 9. Anomaly Score Visualisation

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(scores, linewidth=0.8, alpha=0.7, label='Anomaly Score')
plt.axhline(0, color='red', linestyle='--', linewidth=1, label='Decision Boundary (0)')
plt.scatter(
    np.where(flags == 1)[0], scores[flags == 1],
    color='red', s=20, zorder=5, label=f'Anomaly (n={flags.sum()})'
)
plt.title(
    f'Fine-Tuned Isolation Forest — Heart Rate Anomalies\n'
    f'n_est={bp["n_estimators"]}, '
    f'contamination={bp["contamination"]}, '
    f'max_samples={bp["max_samples"]}'
)
plt.xlabel('Sample Index')
plt.ylabel('Decision Function Score')
plt.legend()
plt.tight_layout()
plt.savefig('final_anomaly_scores.png', dpi=150)
plt.show()
print('Plot saved.')

## 10. Save Results

In [ ]:
df_result = df[feature_cols].copy()
df_result['anomaly_flag']  = flags
df_result['anomaly_score'] = scores

# Attach non-numeric columns back
for col in ['patient_id', 'activity_type', 'timestamp']:
    if col in df.columns:
        df_result[col] = df[col].values

df_result.to_csv('heart_rate_anomaly_results.csv', index=False)

print('Results saved to heart_rate_anomaly_results.csv')
print('\n=== FINAL SUMMARY ===')
print('Broad GridSearch Best Params :', grid_search.best_params_)
print('Fine-Tuned Best Params       :', bp)
print(f'Fine-Tuned CV Score          : {fine_search.best_score_:.4f}')
print(f'Anomalies Flagged            : {flags.sum()} / {len(X)} ({100*flags.mean():.2f}%)')
df_result.head(10)